# K06_03 – Random Forest mit Moons (Student)

## Lernziele
- einen **unregulierten Entscheidungsbaum**, einen **regulierten Entscheidungsbaum** und einen **Random Forest** vergleichen
- Overfitting bei Einzelbäumen erkennen
- verstehen, warum Ensembles oft robuster generalisieren
- den Unterschied zwischen **einzelnem Testsplit** und **wiederholter Cross-Validation** erklären

## Didaktischer Hinweis

In diesem Notebook vergleichen wir **drei Verfahren**:

1. **Baum unreguliert**
2. **Baum reguliert**
3. **Random Forest**

Dadurch wird die zentrale Vorlesungsbotschaft klarer sichtbar:

- unregulierte Einzelbäume sind oft sehr flexibel und overfitten leicht
- regulierte Bäume sind robuster, aber begrenzter
- Random Forests liefern oft eine gute Balance aus Flexibilität und Stabilität

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    cross_validate
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 1. Datensatz erzeugen

Wir verwenden den `make_moons`-Datensatz mit etwas mehr Daten und etwas mehr Rauschen als zuvor.

Warum?
- Die Unterschiede zwischen den Verfahren werden dadurch oft sichtbarer.
- Das Beispiel wird robuster für die Vorlesung.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Two-Moons-Datensatz")
plt.show()

print("Shape von X:", X.shape)
print("Shape von y:", y.shape)

## 2. Trainings- und Testdaten

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 3. Drei Modelle definieren

In [ ]:
models = [
    ("Baum unreguliert", DecisionTreeClassifier(random_state=42)),
    ("Baum reguliert", DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=300, random_state=42))
]

## 4. Erster Vergleich auf einem einzelnen Train/Test-Split

In [ ]:
split_results = []
for name, clf in models:
    clf.fit(X_train, y_train)
    split_results.append({
        "Verfahren": name,
        "Train-Accuracy": clf.score(X_train, y_train),
        "Test-Accuracy": clf.score(X_test, y_test)
    })

split_df = pd.DataFrame(split_results).round(4)
split_df

## 5. Warum ein einzelner Testsplit noch nicht reicht

Der erste Vergleich ist nützlich, aber noch nicht belastbar genug.

Gründe:
- Das Ergebnis hängt immer auch von der **konkreten Datenaufteilung** ab.
- Ein einzelner Split kann einem Verfahren zufällig helfen oder schaden.
- Für die Vorlesung ist deshalb eine **robustere Bewertungsmethode** sinnvoll.

Wir verwenden daher jetzt eine **Repeated Cross-Validation**:
- 10 Folds
- 5 Wiederholungen

Dadurch wird der Vergleich stabiler und weniger zufallsabhängig.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)

cv_results = []
for name, clf in models:
    scores = cross_validate(
        clf,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    cv_results.append({
        "Verfahren": name,
        "CV-Accuracy Mittelwert": scores["test_score"].mean(),
        "CV-Accuracy Std": scores["test_score"].std(),
        "Train-Accuracy Mittelwert": scores["train_score"].mean()
    })

cv_df = pd.DataFrame(cv_results).round(4)
cv_df.sort_values("CV-Accuracy Mittelwert", ascending=False)

## 6. Interpretation der Ergebnisse

In diesem Vergleich betrachten wir drei Verfahren:

1. einen **unregulierten Entscheidungsbaum**
2. einen **regulierten Entscheidungsbaum**
3. einen **Random Forest**

Typische Beobachtung:

- Der **unregulierte Baum** erreicht oft die höchste Trainingsgüte, zeigt aber die stärksten Hinweise auf Overfitting.
- Der **regulierte Baum** ist robuster, aber in seiner Flexibilität begrenzt.
- Der **Random Forest** erreicht häufig die beste oder eine sehr gute Testleistung bei gleichzeitig robusterem Verhalten.

Wichtig:
Für die Vorlesung ist **nicht entscheidend**, dass der Random Forest in jeder einzelnen Zahl klar gewinnt.
Wichtiger ist die methodische Einsicht:

> Einzelbäume sind instabiler und anfälliger für Overfitting, während Ensembles häufig robuster generalisieren.

## 7. Mini-Übungen

1. Welches Modell zeigt die stärksten Hinweise auf Overfitting?
2. Was verändert die Regularisierung am Einzelbaum?
3. Warum ist die wiederholte Cross-Validation hier didaktisch überzeugender als ein einzelner Testsplit?

## 8. Fazit

- Der **unregulierte Baum** ist sehr flexibel, aber anfällig für Overfitting.
- Der **regulierte Baum** ist kontrollierter, aber einfacher.
- Der **Random Forest** ist oft robuster und generalisiert stabiler.
- Für einen fairen Vergleich ist die **Repeated Cross-Validation** eine sehr gute Grundlage.